# Tutorial: Functions Workflow with AgentGrid SDKs
- Author: Shridhar Kini
- To Securely Run: `jupyter notebook password` to generate onetime password for secure access
- To Run at root directory of the repo: `jupyter notebook --allow-root --port 9999 --ip=0.0.0.0` 
- To Clear Outputs: Use `jupyter nbconvert --clear-output --inplace Simple_workflow.ipynb` 

This walkthrough demonstrates how to build and orchestrate a  workflow of agents using the **AgentGrid SDKs** and **AgentGrid Workflow Engine** and define the Behaviour of an agent with other system like **Functions**. We focus on a **Code Generation and Testing Workflow**, where a behaviour is attached to agent with the help of 3 functions connected in sequential manner such that Agents can use the **Reusable Functions** and **Graph of Functions**.

# [ServiceGrid](https://github.com/opencyber-space/Servicegr.id)
```
                  ServiceGrid
                    __|__
                   |     |
          Tools Bank      Functions Bank
         |- Python Code   |- Statefull Functions(Using K8s Deployment)
         |- Binary Code   |- Stateless Jobs(Using K8s Jobs)
```
**Note**:
- Tools will be run within Agent Code i.e it shares CPU Cycles, RAM, Disk of the Agent Pod
- Functions will be run outside of Agent Code i.e CPU Cycles, RAM and Disk are not from Agent Pod.

## 1. Directory Structure
Understanding the files in this demonstration:
* **`nodes/`**: Contains the Python code for each agent (`agent_behavioral_code_creator.py`, `agent_behavioral_reviewer.py`).
* **`spec/`**: Contains agent registration JSON specifications (`agent_behavioral_code_creator.json`, etc.) and bash scripts for agent-level operations (registering, deploying, removing, unregistering).
* **`deploy_run/`**: Contains the `workflow_spec.json` (which defines the data flow between the agents), the Streamlit dashboard code (`streamlit_app.py`, `structure`), and the bash scripts for workflow orchestration and inference.
* **`tools/`**: Containes files related to codes of the tools, its registration spec files, deployment of tools etc



## 2. Register and Build Individual Agents
Before agents can be deployed, they must be registered to the system and packaged as Docker containers.

### Register the agents

In [ ]:
%%bash
(
  cd ../spec/
  bash register_agents.sh
)

### Build the docker images and push them to the repository

In [ ]:
%%bash
(
  cd ../../
  bash build_toolsusage_and_push.bash
)

## 3. Deploy the Behavioral Functions

### 3.1. Zip the Functions Files

In [ ]:
%%bash
(
  cd ../tools/commit-scribe
  bash build.sh
  cd ../log-detective
  bash build.sh
  cd ../schema-forge
  bash build.sh
)

### 3.2. Upload the Zipped Files

In [ ]:
%%bash
(
  cd ../tools/commit-scribe
  bash upload.sh
  cd ../log-detective
  bash upload.sh
  cd ../schema-forge
  bash upload.sh
)

### 3.3. Get API Calls

In [ ]:
%%bash
(
  cd ../tools/commit-scribe
  bash get.sh
  cd ../log-detective
  bash get.sh
  cd ../schema-forge
  bash get.sh
)

## 4. Deploy Agents
Once the images are built and pushed, deploy the agents into the cluster so they are actively waiting for tasks.

### Deploy the agents into the cluster

In [ ]:
%%bash
(
  cd ../spec/
  bash deploy_agents.sh
)

### Verify that the agents are running

In [ ]:
!kubectl get pods -n agents

## 5. Register and Deploy the Workflow
The `workflow_spec.json` defines how the outputs of one agent map dynamically to the inputs of the next agent. We need to register this spec and deploy it.

### Register the workflow

In [ ]:
%%bash
(
  bash register_workflow.sh
)

### Deploy the workflow to establish connections
This creates a central coordinator for the workflow which acts like a Controller co-ordinating between different agents based on workflow defined. It provides API URLs for interacting with input or any other task of workflow

In [ ]:
%%bash
(
  bash deploy_workflow.sh
)

### Verify that the workflow pods are running

In [ ]:
!kubectl get pods -n workflows

## 6. Run the Workflow Demo
With everything deployed, start the Streamlit Dashboard to visualize the flow, and feed the initial payload into the first agent.

### Run the Streamlit app in the background

In [ ]:
%%bash
(
  source ../../../venv/bin/activate
  nohup bash run_streamlit_app.bash > streamlit.log 2>&1 &
  echo "Dashboard starting in background..."
)

Dashboard starting in background...


### Trigger the inference by sending a contract to the workflow inbox

In [ ]:
%%bash
(
  bash workflow_input.sh
)

## 9. Clean Up and Destroy Resources
Once the demonstration is complete, cleanly tear down the resources.

### Remove workflow connections and coordinator pods

In [ ]:
%%bash
(
  bash remove_workflow.sh
)

### Unregister the workflow entirely

In [ ]:
%%bash
(
  bash unregister_workflow.sh
)

### Remove the agent pods

In [ ]:
%%bash
(
  cd ../spec
  bash remove_agents.sh
)

### Unregister the individual agents from the AgentGrid system

In [ ]:
%%bash
(
  cd ../spec
  bash unregister_agents.sh
)